In [ ]:
!pip install pandas plotly dash jupyter-dash

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.7/101.7 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.0/228.0 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 59.7 MB/s eta 0:00:00
  Attempting uninstall: Werkzeug
    Found existing installation: Werkzeug 3.1.3
    Uninstalling Werkzeug-3.1.3:
      Successfully uninstalled Werkzeug-3.1.3
  Attempting uninstall: Flask
    Found existing installation: Flask 3.1.0
    Uninstalling Flask-3.1.0:
      Successfully uninstalled Flask-3.1.0


In [ ]:
import pandas as pd

In [ ]:
import pandas as pd
import plotly.express as px
from jupyter_dash import JupyterDash
from dash import Dash, dcc, html, Input, Output

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving archive (1).zip to archive (1).zip


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving archive (1).zip to archive (1) (1).zip


In [ ]:
import zipfile
import os
zip_filename = "archive (1).zip"

with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
    zip_ref.extractall("dataset")

extracted_files = os.listdir("dataset")
print(extracted_files)

['Prime TV Shows Data set.csv']


In [ ]:
df = pd.read_csv('dataset/Prime TV Shows Data set.csv', encoding='ISO-8859-1')

In [ ]:
print("Columns in the dataset:")
print(df.columns.tolist())

print("\nFirst 8 rows:")
display(df.head())

Columns in the dataset:
['S.no.', 'Name of the show', 'Year of release', 'No of seasons available', 'Language', 'Genre', 'IMDb rating', 'Age of viewers']

First 8 rows:


,S.no.,Name of the show,Year of release,No of seasons available,Language,Genre,IMDb rating,Age of viewers
0,1,Pataal Lok,2020.0,1.0,Hindi,Drama,7.5,18+
1,2,Upload,2020.0,1.0,English,Sci-fi comedy,8.1,16+
2,3,The Marvelous Mrs. Maisel,2017.0,3.0,English,"Drama, Comedy",8.7,16+
3,4,Four More Shots Please,2019.0,2.0,Hindi,"Drama, Comedy",5.3,18+
4,5,Fleabag,2016.0,2.0,English,Comedy,8.7,18+


In [ ]:
print("Missing values before cleaning:")
print(df.isnull().sum())

df['IMDb rating'].fillna(df['IMDb rating'].mean(), inplace=True)
df.dropna(subset=['Name of the show', 'Genre', 'Language'], inplace=True)

df['Year of release'] = df['Year of release'].astype(int)
df['No of seasons available'] = df['No of seasons available'].astype(int)

print(f"Total duplicates found: {df.duplicated().sum()}")
df.drop_duplicates(inplace=True)

df['Genre'] = df['Genre'].str.lower()
df['Language'] = df['Language'].str.lower()
df['Age of viewers'] = df['Age of viewers'].str.lower()

df.to_csv('cleaned_prime_shows.csv', index=False)
print("Cleaning complete! Your file is ready as 'cleaned_prime_shows.csv'")

Missing values before cleaning:
S.no.                        0
Name of the show            11
Year of release             11
No of seasons available     11
Language                    11
Genre                       11
IMDb rating                222
Age of viewers              11
dtype: int64
Total duplicates found: 0
Cleaning complete! Your file is ready as 'cleaned_prime_shows.csv'


<ipython-input-25-cc7d4c4085ec>:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['IMDb rating'].fillna(df['IMDb rating'].mean(), inplace=True)


In [ ]:
import matplotlib.pyplot as plt


In [ ]:
df = pd.read_csv('cleaned_prime_shows.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'cleaned_prime_shows.csv'

In [ ]:
df = pd.read_csv('cleaned_prime_shows.csv')

# Create figures ahead of layout
# Line Chart: IMDb rating over time
line_fig = px.line(df, x='Year of release', y='IMDb rating', color='Genre',
                   title='IMDb Ratings Over Time', labels={'Year of release': 'Year', 'IMDb rating': 'IMDb Rating'})

# Bar Chart: Count of shows by genre
genre_counts = df['Genre'].value_counts().reset_index()
genre_counts.columns = ['Genre', 'Count']
bar_fig = px.bar(genre_counts, x='Genre', y='Count', title='Number of Shows by Genre')

# Heatmap: Correlation between numerical features
corr_matrix = df[['IMDb rating', 'Year of release', 'No of seasons available']].corr()
heatmap_fig = px.imshow(corr_matrix, text_auto=True, title='Correlation Heatmap')

# Histogram: IMDb rating distribution
hist_fig = px.histogram(df, x='IMDb rating', title='IMDb Rating Distribution')

# Initialize Dash app
app = JupyterDash(__name__)

app.layout = html.Div([
    html.H1("Prime Video Shows Dashboard", style={'textAlign': 'center'}),

    dcc.Dropdown(
        id='genre-dropdown',
        options=[{'label': genre, 'value': genre} for genre in df['Genre'].unique()],
        value=df['Genre'].unique()[0],
        clearable=False,
        placeholder='Select Genre'
    ),

    dcc.Graph(id='line-chart', figure=line_fig),
    dcc.Graph(id='bar-chart', figure=bar_fig),
    dcc.Graph(id='heatmap', figure=heatmap_fig),
    dcc.Graph(id='histogram', figure=hist_fig)
])

# Callbacks for interactivity
@app.callback(
    Output('line-chart', 'figure'),
    Input('genre-dropdown', 'value')
)
def update_line_chart(selected_genre):
    filtered_df = df[df['Genre'] == selected_genre]
    fig = px.line(filtered_df, x='Year of release', y='IMDb rating',
                  title=f'IMDb Ratings Over Time for {selected_genre}')
    return fig

if __name__ == '__main__':
    app.run_server(mode='inline', debug=True)

FileNotFoundError: [Errno 2] No such file or directory: 'cleaned_prime_shows.csv'